# NEU-DET Surface Defect Detection — YOLO Training (Google Colab)

**Before running:** `Runtime → Change runtime type → Hardware accelerator: T4 GPU`.

Run the cells top to bottom. Each section says what it does and what you should see.

Pipeline: `Roboflow (NEU-DET in YOLO format) → smoke test → full training → validation → download best.pt`

## 1. Verify the GPU
If this prints a table with *Tesla T4* (or similar) you have a GPU. If it errors, fix the runtime type first.

In [ ]:
!nvidia-smi

## 2. Install dependencies
`ultralytics` = the YOLO library. `roboflow` = downloads the dataset in YOLO format.

In [ ]:
%pip install -q -U ultralytics roboflow
import ultralytics, torch
print("ultralytics", ultralytics.__version__)
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 3. Download NEU-DET from Roboflow (YOLO format)

1. Go to https://universe.roboflow.com and search **NEU-DET** (or "NEU surface defect").
2. Open a project, click **Download Dataset**, choose format **YOLOv8** (also works for YOLO11/YOLO26), and select **"show download code"**.
3. Copy the snippet. It contains **your** `api_key`, `workspace`, `project` and version number. Paste those four values below.

Do not commit your API key to GitHub.

In [ ]:
from roboflow import Roboflow

# ---- YOUR ROBOFLOW VALUES ----
API_KEY   = "UT0I0aqrf9c1roj5qFo7"   # regenerate it first — the old one was exposed in the screenshot
WORKSPACE = "zxc-ry4eb"
PROJECT   = "nue-snd75"
VERSION   = 1
# ------------------------------

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")

DATASET_DIR = dataset.location
print("Dataset downloaded to:", DATASET_DIR)

: 

## 4. Inspect the dataset — do not assume its structure

We check:
- `data.yaml` → class names and number of classes (`nc`)
- folder layout → `train/`, `valid/`, `test/`, each with `images/` and `labels/`
- one label file → each line is `class_id x_center y_center width height` (normalised 0–1)

**Expected for NEU-DET:** 6 classes — crazing, inclusion, patches, pitted_surface, rolled-in_scale, scratches. If your Roboflow version differs, the printed `names` list is the truth.

In [ ]:
import os, yaml, glob

YAML_PATH = os.path.join(DATASET_DIR, "data.yaml")
with open(YAML_PATH) as f:
    data_cfg = yaml.safe_load(f)

print("Classes (nc):", data_cfg["nc"])
print("Names:", data_cfg["names"])
print()

for split in ["train", "valid", "test"]:
    imgs = glob.glob(os.path.join(DATASET_DIR, split, "images", "*"))
    lbls = glob.glob(os.path.join(DATASET_DIR, split, "labels", "*.txt"))
    print(f"{split:6s}  images={len(imgs):5d}  labels={len(lbls):5d}")

# Peek at one label file
sample_label = sorted(glob.glob(os.path.join(DATASET_DIR, "train", "labels", "*.txt")))[0]
print("\nSample label file:", os.path.basename(sample_label))
print(open(sample_label).read())
print("Format: class_id  x_center  y_center  width  height  (all normalised 0-1)")

### 4b. Fix data.yaml paths
Roboflow writes relative paths that sometimes break in Colab. We rewrite them as absolute paths so Ultralytics always finds the folders.

In [ ]:
data_cfg["train"] = os.path.join(DATASET_DIR, "train", "images")
data_cfg["val"]   = os.path.join(DATASET_DIR, "valid", "images")
data_cfg["test"]  = os.path.join(DATASET_DIR, "test",  "images")
with open(YAML_PATH, "w") as f:
    yaml.safe_dump(data_cfg, f)
print(open(YAML_PATH).read())

## 5. Visualise a few images with their boxes
If boxes sit on the defects, the labels are correct. If boxes are misplaced, stop and check the dataset.

In [ ]:
import cv2, random
import matplotlib.pyplot as plt

names = data_cfg["names"]
img_paths = sorted(glob.glob(os.path.join(DATASET_DIR, "train", "images", "*")))
random.seed(0)
picks = random.sample(img_paths, 6)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, p in zip(axes.ravel(), picks):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl = p.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    if os.path.exists(lbl):
        for line in open(lbl):
            c, xc, yc, bw, bh = map(float, line.split())
            x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
            x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(img, names[int(c)], (x1, max(y1 - 4, 10)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)
    ax.imshow(img); ax.set_title(os.path.basename(p)[:25]); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Smoke test — 1 epoch

Purpose: prove the whole pipeline runs (paths, labels, GPU) before spending 30+ minutes. Takes ~1–2 min.

- `yolo26n.pt` = smallest pretrained YOLO26 model (fast, ~5 MB weights, fine for a demo). If it fails to download on your Ultralytics version, use `yolo11n.pt`.
- `imgsz=256` — NEU-DET images are 200×200, so a large size wastes compute.

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolo26n.pt"   # fallback: "yolo11n.pt"

model = YOLO(BASE_MODEL)
model.train(data=YAML_PATH, epochs=1, imgsz=256, batch=32, project="runs/detect", name="smoke", exist_ok=True)
print("Smoke test finished. Check above for 'Results saved to runs/detect/smoke'.")

## 7. Full training

Key settings, explained:
- **epochs** — one pass over all training images. 60 with `patience=15` means training stops early if validation mAP stops improving for 15 epochs.
- **batch** — images per step. 32 fits comfortably on a T4 at 256px.
- **imgsz** — images are resized to this. 256 (multiple of 32) is close to NEU-DET's native 200.
- **augmentation** — Ultralytics applies random flips, HSV colour shifts, translation, scaling and mosaic by default. This is how the model learns to handle lighting/orientation variation.
- **learning** — every step the model predicts boxes, compares them to the labels (loss), and adjusts its weights slightly.
- **validation** — after each epoch the model is scored on images it never trained on; `best.pt` is the checkpoint with the best validation mAP.

Expect ~20–40 min on a T4. Do not close the tab.

In [ ]:
model = YOLO(BASE_MODEL)
results = model.train(
    data=YAML_PATH,
    epochs=60,
    patience=15,
    imgsz=256,
    batch=32,
    project="runs/detect",
    name="neu_det",
    exist_ok=True,
    seed=0,
)
BEST = "runs/detect/neu_det/weights/best.pt"
print("best.pt at:", BEST, "| exists:", os.path.exists(BEST))

## 8. Evaluation — real numbers only

Run validation with `best.pt` and read the metrics. **Write these into README.md.** Do not report numbers you did not see here.

- **Precision** — of everything the model called a defect, how much really was a defect. Low precision = many false alarms (false positives).
- **Recall** — of all real defects, how many the model found. Low recall = missed defects (false negatives). In a factory, missed defects are usually worse than false alarms.
- **mAP50** — average detection quality when a box counts as correct if it overlaps the true box by ≥50%. The headline number.
- **mAP50-95** — same, averaged over stricter overlap levels 50%…95%. Always lower; rewards tight boxes.

**Training vs validation vs real-world:** training metrics come from images the model memorised (optimistic). Validation metrics use held-out images (honest estimate). Real-world performance on a different camera/lighting/steel grade is unknown until tested.

In [ ]:
best_model = YOLO(BEST)
metrics = best_model.val(data=YAML_PATH, split="val", imgsz=256, plots=True)

print("\n=== VALIDATION RESULTS (copy into README) ===")
print(f"Precision : {metrics.box.mp:.3f}")
print(f"Recall    : {metrics.box.mr:.3f}")
print(f"mAP50     : {metrics.box.map50:.3f}")
print(f"mAP50-95  : {metrics.box.map:.3f}")
print("\nPer-class mAP50-95:")
for i, ap in zip(metrics.box.ap_class_index, metrics.box.maps[metrics.box.ap_class_index]):
    print(f"  {names[int(i)]:18s} {ap:.3f}")

### 8b. Confusion matrix and training curves
The confusion matrix shows which classes get mixed up (e.g. crazing vs rolled-in_scale). `results.png` shows loss going down and mAP going up over epochs.

In [ ]:
from IPython.display import Image, display
VAL_DIR = str(metrics.save_dir)
for fname in ["confusion_matrix_normalized.png"]:
    p = os.path.join(VAL_DIR, fname)
    if os.path.exists(p): display(Image(p, width=700))
p = "runs/detect/neu_det/results.png"
if os.path.exists(p): display(Image(p, width=900))

### 8c. Choosing a threshold (optional, 2 min)

The app uses 0.25 as a **demo threshold**. This cell shows precision and recall at several thresholds so you can justify a value to judges. Pick a threshold where recall stays high (few missed defects) without precision collapsing.

In [ ]:
for conf in [0.15, 0.25, 0.35, 0.5]:
    m = best_model.val(data=YAML_PATH, split="val", imgsz=256, conf=conf, plots=False, verbose=False)
    print(f"conf={conf:.2f}  precision={m.box.mp:.3f}  recall={m.box.mr:.3f}  mAP50={m.box.map50:.3f}")

## 9. Test the model on unseen test images
This is exactly what the Gradio app will do. Boxes with class + confidence should appear on the defects.

In [ ]:
test_imgs = sorted(glob.glob(os.path.join(DATASET_DIR, "test", "images", "*")))[:6]
res = best_model.predict(test_imgs, conf=0.25, imgsz=256, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, r in zip(axes.ravel(), res):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    dets = [f"{r.names[int(b.cls[0])]} {float(b.conf[0]):.2f}" for b in r.boxes]
    ax.set_title(", ".join(dets) if dets else "no detection", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 10. Save best.pt (do this before the Colab session dies)

Two copies: Google Drive (safe) and a direct download (put it in `model/best.pt` in the project).
Also grab a few test images for `demo/sample_images/`.

In [ ]:
from google.colab import drive, files
import shutil

drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/neu_det_hackathon", exist_ok=True)
shutil.copy(BEST, "/content/drive/MyDrive/neu_det_hackathon/best.pt")
shutil.copy("runs/detect/neu_det/results.png", "/content/drive/MyDrive/neu_det_hackathon/results.png")
print("Saved to Google Drive:", os.listdir("/content/drive/MyDrive/neu_det_hackathon"))

# Zip a few test images for the demo
os.makedirs("demo_samples", exist_ok=True)
for p in test_imgs:
    shutil.copy(p, "demo_samples")
shutil.make_archive("demo_samples", "zip", "demo_samples")

files.download(BEST)
files.download("demo_samples.zip")